# Gemma 4 31B Assistant-V7 Byte-Head Speculative Drafter (`BHDR` v1 — Colab Pro GPU)

Builds an ultra-compact `~57–108 MB` (vs `334 MB` V5/V6) speculative drafter for `run31b` on Raspberry Pi 4:
1. Strips `model.embed_tokens.weight` (`~268 MB` BF16 / `~136 MB` Q4) by repacking `google/gemma-4-31B-it-assistant` into embed-free `gemma4_31b_assistant_v7_body_q4.t16c` (`~107 MB`) and 2-bit Row-Major Gauss `gemma4_31b_assistant_v7_body_q2.t16c` (`~56 MB`).
2. Distills a `~1.6 MB` INT8 `ByteHeadGRU` (`gemma4_31b_assistant_v7.bhdr`) using exact single-pass `<eot>` knowledge distillation (`arXiv:2609.12303`, Appendix F.1) over live Assistant hidden states & top-K teacher logit distributions across FineWeb + code text.
3. Saves verified artifacts + SHA-256 receipts to `/content/drive/MyDrive/gemma4_v6_1_hybrid` and auto-unassigns (`COLAB_AUTO_UNASSIGN=1`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
DRIVE_OUT = Path("/content/drive/MyDrive/gemma4_v6_1_hybrid")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "v5-qat-dev", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=True)
subprocess.run(["nvidia-smi"], check=True)

_need_pip = False
try:
    import datasets, huggingface_hub, safetensors, tokenizers
except Exception:
    _need_pip = True
if _need_pip:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "safetensors", "tokenizers", "datasets"], check=True)

try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
        print("Loaded HF_TOKEN from Colab secrets.")
except Exception:
    pass

In [ ]:
import gc
import hashlib
import json
import math
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import hf_hub_download, snapshot_download
from safetensors import safe_open
from tokenizers import Tokenizer

CALIB_DIR = Path("/content/Language-U-Gemma4/experiments/run270_rust/calib/31b")
sys.path.insert(0, str(CALIB_DIR))
import train_assistant_v7_byte_head as bhmod

assert bhmod._run_selftest() == 0, "Byte-Head 6-gate selftest failed"

DRIVE_OUT = Path("/content/drive/MyDrive/gemma4_v6_1_hybrid")
LOCAL_OUT = Path("/content/assistant_v7_out")
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

base_dir = Path(snapshot_download(
    "google/gemma-4-31B-it-qat-w4a16-ct",
    local_dir="/tmp/gemma4_31b",
    ignore_patterns=["*.bin", "*.pt"],
))
ast_dir = Path(snapshot_download(
    "google/gemma-4-31B-it-assistant",
    local_dir="/tmp/gemma4_ast_hf",
    ignore_patterns=["*.bin", "*.pt"],
))
base_sf = base_dir / "model.safetensors"
ast_sf = ast_dir / "model.safetensors"
tok_path = base_dir / "tokenizer.json"
cfg_full = json.loads((base_dir / "config.json").read_text())

# 1. Pack embed-free Q4 (+FWHT) and Q2 Row-Major Assistant bodies (~107 MB and ~56 MB)
body_q4 = LOCAL_OUT / "gemma4_31b_assistant_v7_body_q4.t16c"
body_q2 = LOCAL_OUT / "gemma4_31b_assistant_v7_body_q2.t16c"
bhmod.pack_assistant_body_from_safetensors(ast_sf, body_q4, body_bits=4, block_size=64)
bhmod.pack_assistant_body_from_safetensors(ast_sf, body_q2, body_bits=2, block_size=64)

# Also strip embed from V6.1 assistant if already present in Drive
ast_v61 = DRIVE_OUT / "gemma4_31b_assistant_v6_1.t16c"
body_q8 = LOCAL_OUT / "gemma4_31b_assistant_v7_body_q8.t16c"
if ast_v61.is_file():
    bhmod.repack_assistant_t16c_strip_embed(ast_v61, body_q8)
else:
    bhmod.pack_assistant_body_from_safetensors(ast_sf, body_q8, body_bits=8, block_size=64)

# 2. Verify full 262K Gemma-4 trie
trie = bhmod.BpePrefixTrie.from_tokenizer_json(tok_path)
assert trie.collision_count == 0, f"Unexpected trie collisions: {trie.collision_count}"
print(f"[Trie OK] covered={trie.covered_tokens} nodes={len(trie.node_edges):,} edges={trie.total_edges:,} csr_mb={trie.approx_rust_csr_bytes()/(1024**2):.2f}")

# 3. Authentic GPU Activation Extraction: stream W4A16 L00..L59 + 4-layer Assistant KV sharing
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
COMPUTE_DTYPE = torch.bfloat16 if (DEVICE.startswith("cuda") and torch.cuda.is_bf16_supported()) else torch.float16
tc = cfg_full["text_config"]
N_LAYERS = int(tc["num_hidden_layers"])
D_MOD = int(tc["hidden_size"])
EPS = float(tc["rms_norm_eps"])
ESCALE = math.sqrt(float(D_MOD))
NH = int(tc["num_attention_heads"])
NKV_S = int(tc["num_key_value_heads"])
NKV_F = int(tc["num_global_key_value_heads"])
HD_S = int(tc["head_dim"])
HD_F = int(tc["global_head_dim"])
K_EQ_V = bool(tc.get("attention_k_eq_v", False))
LAYER_TYPES = tc["layer_types"]
RP = tc["rope_parameters"]
THETA_G = float(RP["full_attention"]["rope_theta"])
PART_G = float(RP["full_attention"].get("partial_rotary_factor", 1.0))
THETA_L = float(RP["sliding_attention"]["rope_theta"])
PART_L = float(RP["sliding_attention"].get("partial_rotary_factor", 1.0))
N_SEQ, SEQ_LEN = 48, 256

tk = Tokenizer.from_file(str(tok_path))
ds_fw = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
ds_alp = load_dataset("yahma/alpaca-cleaned", split="train", streaming=True)
fw_buf, chat_buf = [], []
for row in ds_fw:
    text = row["text"].strip()
    if not text:
        continue
    ids_row = tk.encode(text, add_special_tokens=False).ids
    if len(ids_row) >= 32:
        fw_buf.extend(ids_row[:SEQ_LEN])
    if len(fw_buf) >= (N_SEQ // 2) * SEQ_LEN:
        break
for row in ds_alp:
    q_t = (row.get("instruction", "") + ("\n" + row.get("input", "") if row.get("input", "") else "")).strip()
    a_t = row.get("output", "").strip()
    if not q_t or not a_t:
        continue
    turn_txt = f"<|turn>user\n{q_t}<turn|>\n<|turn>model\n<|channel>thought\n<channel|>{a_t}<turn|>\n"
    chat_buf.extend(tk.encode(turn_txt, add_special_tokens=False).ids)
    if len(chat_buf) >= (N_SEQ - N_SEQ // 2) * SEQ_LEN:
        break
n_fw = N_SEQ // 2
seqs_list = [[2] + fw_buf[i * (SEQ_LEN - 1) : (i + 1) * (SEQ_LEN - 1)] for i in range(n_fw)]
seqs_list += [[2] + chat_buf[i * (SEQ_LEN - 1) : (i + 1) * (SEQ_LEN - 1)] for i in range(N_SEQ - n_fw)]
all_seq_ids = np.array(seqs_list, dtype=np.int64)

pos_1d = torch.arange(SEQ_LEN, device=DEVICE, dtype=torch.float32)
def make_rope(theta: float, rot_d: int, hd: int):
    hr = rot_d // 2
    inv_f = 1.0 / (theta ** (torch.arange(0, 2 * hr, 2, device=DEVICE, dtype=torch.float32) / float(hd)))
    ang = pos_1d[:, None] * inv_f[None, :]
    return torch.cos(ang), torch.sin(ang)

def apply_rope(t: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, rot_d: int, hd: int) -> torch.Tensor:
    hr, hh = rot_d // 2, hd // 2
    c = cos[None, :, None, :].to(t.dtype)
    s = sin[None, :, None, :].to(t.dtype)
    x0, x1 = t[..., :hr], t[..., hh : hh + hr]
    r0, r1 = x0 * c - x1 * s, x0 * s + x1 * c
    if hr == hh:
        return torch.cat([r0, r1], dim=-1)
    return torch.cat([r0, t[..., hr:hh], r1, t[..., hh + hr :]], dim=-1)

def rmsn(x: torch.Tensor, w=None) -> torch.Tensor:
    xf = x.float()
    y = xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + EPS)
    return ((y * w.float()) if w is not None else y).to(COMPUTE_DTYPE)

with safe_open(base_sf, framework="pt", device="cpu") as sf:
    sf_keys = set(sf.keys())
    def unpack_proj(pfx: str) -> torch.Tensor:
        if f"{pfx}.weight_packed" in sf_keys:
            wp = sf.get_tensor(f"{pfx}.weight_packed").to(DEVICE)
            ws = sf.get_tensor(f"{pfx}.weight_scale").to(DEVICE, dtype=torch.float32)
            sh = torch.arange(0, 32, 4, device=DEVICE, dtype=torch.int32)
            nibs = ((wp.unsqueeze(-1) >> sh) & 0xF) - 8
            return (nibs.to(torch.float32).reshape(wp.shape[0], -1, 32) * ws.unsqueeze(-1)).reshape(wp.shape[0], -1).to(COMPUTE_DTYPE)
        return sf.get_tensor(f"{pfx}.weight").to(DEVICE, dtype=COMPUTE_DTYPE)

    emb_31b = sf.get_tensor("model.language_model.embed_tokens.weight")
    X_cpu = (emb_31b[torch.from_numpy(all_seq_ids).long()].float() * ESCALE).to(COMPUTE_DTYPE).cpu()
    fn_w_31b = sf.get_tensor("model.language_model.norm.weight").to(DEVICE, dtype=torch.float32)
    cap_kv58 = None
    cap_kv59 = None
    print(f"[Activations] Streaming {N_LAYERS} W4A16 teacher layers over {N_SEQ}x{SEQ_LEN} = {N_SEQ * SEQ_LEN:,} tokens...", flush=True)
    for l_idx in range(N_LAYERS):
        p = f"model.language_model.layers.{l_idx}."
        is_full = LAYER_TYPES[l_idx] == "full_attention"
        nkv = NKV_F if is_full else NKV_S
        hd = HD_F if is_full else HD_S
        rot_d = int(round(hd * (PART_G if is_full else PART_L)))
        cos_r, sin_r = make_rope(THETA_G if is_full else THETA_L, rot_d, hd)
        gqa_rep = NH // nkv
        w_q = unpack_proj(p + "self_attn.q_proj")
        w_k = unpack_proj(p + "self_attn.k_proj")
        w_v = None if (is_full and K_EQ_V) else unpack_proj(p + "self_attn.v_proj")
        w_o = unpack_proj(p + "self_attn.o_proj")
        w_g = unpack_proj(p + "mlp.gate_proj")
        w_u = unpack_proj(p + "mlp.up_proj")
        w_d = unpack_proj(p + "mlp.down_proj")
        n_in = sf.get_tensor(p + "input_layernorm.weight").to(DEVICE, dtype=torch.float32)
        n_qn = sf.get_tensor(p + "self_attn.q_norm.weight").to(DEVICE, dtype=torch.float32)
        n_kn = sf.get_tensor(p + "self_attn.k_norm.weight").to(DEVICE, dtype=torch.float32)
        n_pa = sf.get_tensor(p + "post_attention_layernorm.weight").to(DEVICE, dtype=torch.float32)
        n_pf = sf.get_tensor(p + "pre_feedforward_layernorm.weight").to(DEVICE, dtype=torch.float32)
        n_pff = sf.get_tensor(p + "post_feedforward_layernorm.weight").to(DEVICE, dtype=torch.float32)
        lsc = float(sf.get_tensor(p + "layer_scalar").float().item())

        X_next = torch.empty_like(X_cpu)
        k_buf = [] if l_idx in (58, 59) else None
        v_buf = [] if l_idx in (58, 59) else None
        with torch.no_grad():
            for b0 in range(0, N_SEQ, 16):
                b1 = min(b0 + 16, N_SEQ)
                xb = X_cpu[b0:b1].to(DEVICE)
                inp = rmsn(xb, n_in)
                q = F.linear(inp, w_q).view(b1 - b0, SEQ_LEN, NH, hd)
                k = F.linear(inp, w_k).view(b1 - b0, SEQ_LEN, nkv, hd)
                v = k.clone() if (is_full and K_EQ_V) else F.linear(inp, w_v).view(b1 - b0, SEQ_LEN, nkv, hd)
                qn = apply_rope(rmsn(q, n_qn), cos_r, sin_r, rot_d, hd).transpose(1, 2)
                kn = apply_rope(rmsn(k, n_kn), cos_r, sin_r, rot_d, hd).transpose(1, 2)
                vn = rmsn(v, None).transpose(1, 2)
                if l_idx in (58, 59):
                    k_buf.append(kn.cpu())
                    v_buf.append(vn.cpu())
                ctx = F.scaled_dot_product_attention(qn, kn.repeat_interleave(gqa_rep, 1), vn.repeat_interleave(gqa_rep, 1), is_causal=True, scale=1.0)
                ot = F.linear(ctx.transpose(1, 2).contiguous().view(b1 - b0, SEQ_LEN, NH * hd), w_o)
                xm = xb.float() + rmsn(ot, n_pa).float()
                hm = rmsn(xm, n_pf)
                act = (F.gelu(F.linear(hm, w_g).float(), approximate="tanh") * F.linear(hm, w_u).float()).to(COMPUTE_DTYPE)
                X_next[b0:b1] = ((xm + rmsn(F.linear(act, w_d), n_pff).float()) * lsc).to(COMPUTE_DTYPE).cpu()
        X_cpu = X_next
        if l_idx == 58:
            cap_kv58 = (torch.cat(k_buf, dim=0), torch.cat(v_buf, dim=0))
        elif l_idx == 59:
            cap_kv59 = (torch.cat(k_buf, dim=0), torch.cat(v_buf, dim=0))
        del w_q, w_k, w_v, w_o, w_g, w_u, w_d
        torch.cuda.empty_cache()
        if l_idx % 15 == 14 or l_idx == N_LAYERS - 1:
            print(f"  [Teacher Stream] L{l_idx+1:02d}/{N_LAYERS} done", flush=True)

    with torch.no_grad():
        h_59_final = rmsn(X_cpu.to(DEVICE), fn_w_31b).cpu()

# Run 4-layer Assistant KV-sharing backbone on real 31B hidden states + L58/L59 KV -> genuine h_final [N, 1024]
with safe_open(ast_sf, framework="pt", device="cpu") as asf:
    pre_w = asf.get_tensor("pre_projection.weight").to(DEVICE, dtype=COMPUTE_DTYPE)
    fn_ast = asf.get_tensor("model.norm.weight").to(DEVICE, dtype=torch.float32)
    cr_l, sr_l = make_rope(THETA_L, int(round(HD_S * PART_L)), HD_S)
    cr_g, sr_g = make_rope(THETA_G, int(round(HD_F * PART_G)), HD_F)
    h_ast_chunks = []
    with torch.no_grad():
        Sm1 = SEQ_LEN - 1
        q_mask = torch.tril(torch.ones((Sm1, SEQ_LEN), device=DEVICE, dtype=torch.bool), diagonal=0).unsqueeze(0).unsqueeze(0)
        for b0 in range(0, N_SEQ, 16):
            b1 = min(b0 + 16, N_SEQ)
            tok_e = (emb_31b[torch.from_numpy(all_seq_ids[b0:b1, 1:]).long()].float() * ESCALE).to(DEVICE, dtype=COMPUTE_DTYPE)
            back_h = h_59_final[b0:b1, :-1].to(DEVICE, dtype=COMPUTE_DTYPE)
            st = F.linear(torch.cat([tok_e, back_h], dim=-1), pre_w).float()
            for l_a in range(4):
                pa = f"model.layers.{l_a}."
                is_f = (l_a == 3)
                hd_a, nkv_a = (HD_F, NKV_F) if is_f else (HD_S, NKV_S)
                rot_a = int(round(HD_F * PART_G)) if is_f else int(round(HD_S * PART_L))
                gqa_a = NH // nkv_a
                inp = rmsn(st, asf.get_tensor(pa + "input_layernorm.weight").to(DEVICE, dtype=torch.float32))
                q = F.linear(inp, asf.get_tensor(pa + "self_attn.q_proj.weight").to(DEVICE, dtype=COMPUTE_DTYPE)).view(b1 - b0, Sm1, NH, hd_a)
                cr = (cr_g if is_f else cr_l)[1 : Sm1 + 1]
                sr = (sr_g if is_f else sr_l)[1 : Sm1 + 1]
                qn = apply_rope(rmsn(q, asf.get_tensor(pa + "self_attn.q_norm.weight").to(DEVICE, dtype=torch.float32)), cr, sr, rot_a, hd_a).transpose(1, 2)
                kv_src = cap_kv59 if is_f else cap_kv58
                kk = kv_src[0][b0:b1].to(DEVICE, dtype=COMPUTE_DTYPE).repeat_interleave(gqa_a, dim=1)
                vv = kv_src[1][b0:b1].to(DEVICE, dtype=COMPUTE_DTYPE).repeat_interleave(gqa_a, dim=1)
                ctx = F.scaled_dot_product_attention(qn, kk, vv, attn_mask=q_mask, scale=1.0)
                st = st + rmsn(F.linear(ctx.transpose(1, 2).contiguous().view(b1 - b0, Sm1, NH * hd_a), asf.get_tensor(pa + "self_attn.o_proj.weight").to(DEVICE, dtype=COMPUTE_DTYPE)), asf.get_tensor(pa + "post_attention_layernorm.weight").to(DEVICE, dtype=torch.float32)).float()
                hm = rmsn(st, asf.get_tensor(pa + "pre_feedforward_layernorm.weight").to(DEVICE, dtype=torch.float32))
                act = (F.gelu(F.linear(hm, asf.get_tensor(pa + "mlp.gate_proj.weight").to(DEVICE, dtype=COMPUTE_DTYPE)).float(), approximate="tanh") * F.linear(hm, asf.get_tensor(pa + "mlp.up_proj.weight").to(DEVICE, dtype=COMPUTE_DTYPE)).float()).to(COMPUTE_DTYPE)
                lsc_a = float(asf.get_tensor(pa + "layer_scalar").float().item())
                st = (st + rmsn(F.linear(act, asf.get_tensor(pa + "mlp.down_proj.weight").to(DEVICE, dtype=COMPUTE_DTYPE)), asf.get_tensor(pa + "post_feedforward_layernorm.weight").to(DEVICE, dtype=torch.float32)).float()) * lsc_a
            h_ast_chunks.append(rmsn(st, fn_ast).float().cpu())
del cap_kv58, cap_kv59, pre_w
torch.cuda.empty_cache()

# Extract teacher top-64 exact probability distributions from real 31B h_59_final[:, 1:-1] (predicting all_seq_ids[:, 2:])
H_ast_all = torch.cat(h_ast_chunks, dim=0)[:, :-1, :].reshape(-1, 1024)
H_teach_all = h_59_final[:, 1:-1, :].reshape(-1, D_MOD)
gold_flat = torch.from_numpy(all_seq_ids[:, 2:].reshape(-1)).long()
emb_31b_gpu = emb_31b.to(DEVICE, dtype=COMPUTE_DTYPE)
top_ids_t = torch.empty((H_teach_all.shape[0], 64), dtype=torch.int32)
top_probs_t = torch.empty((H_teach_all.shape[0], 64), dtype=torch.float32)
with torch.no_grad():
    for i0 in range(0, H_teach_all.shape[0], 256):
        i1 = min(i0 + 256, H_teach_all.shape[0])
        ht = H_teach_all[i0:i1].to(DEVICE, dtype=COMPUTE_DTYPE)
        logits_b = 30.0 * torch.tanh((ht @ emb_31b_gpu.T).float() / 30.0)
        tv, ti = torch.topk(logits_b, k=64, dim=-1)
        top_probs_t[i0:i1] = F.softmax(tv, dim=-1).cpu()
        top_ids_t[i0:i1] = ti.to(torch.int32).cpu()
del emb_31b, emb_31b_gpu, X_cpu, h_59_final, H_teach_all
gc.collect()
torch.cuda.empty_cache()

act_bundle = LOCAL_OUT / "assistant_v7_activations.pt"
torch.save({"h_final": H_ast_all.half(), "gold_ids": gold_flat, "top_ids": top_ids_t, "top_probs": top_probs_t}, act_bundle)
print(f"[Activations Saved] {act_bundle} (samples={H_ast_all.shape[0]:,})", flush=True)

bhdr_out = LOCAL_OUT / "gemma4_31b_assistant_v7.bhdr"
bh_rpt = bhmod.train_from_activations(
    act_bundle,
    tok_path,
    bhdr_out,
    d_byte=256,
    max_bytes=32,
    steps=1200,
    batch_size=128,
    lr=3e-3,
    device=DEVICE,
)
metrics = bh_rpt["metrics"]
bhdr_sz = bh_rpt["bytes"]
print(f"[Assistant-V7 BHDR] bytes={bhdr_sz:,} ({bhdr_sz/(1024**2):.2f} MB) metrics={metrics}")

# Verify exported INT8 BHDR quantized beam top-1 accuracy on 64 held-out samples
bh_loaded = bhmod.load_bhdr_v1(bhdr_out)
val_h = H_ast_all[-64:].numpy().astype(np.float32)
val_g = gold_flat[-64:].tolist()
beam_hits = sum(1 for hv, gv in zip(val_h, val_g) if trie.tok_syms[gv] and bhmod.decode_bhdr_np_beam(bh_loaded, trie, hv, beam_width=2)[0] == gv)
metrics["heldout_beam_top1_acc"] = round(beam_hits / 64.0, 4)
print(f"[BHDR Quantized Eval] heldout_beam_top1_acc={metrics['heldout_beam_top1_acc']:.4f}")

# 4. Persist to Drive atomically with SHA-256
manifest = {"metrics": metrics, "artifacts": {}}
for fp in [bhdr_out, body_q2, body_q4, body_q8]:
    dst = DRIVE_OUT / fp.name
    tmp = DRIVE_OUT / (fp.name + ".tmp")
    shutil.copy2(fp, tmp)
    os.replace(tmp, dst)
    digest = hashlib.sha256(fp.read_bytes()).hexdigest()
    manifest["artifacts"][fp.name] = {"bytes": fp.stat().st_size, "sha256": digest}
    print(f"[Synced Drive] {fp.name}: {fp.stat().st_size:,} B  sha256={digest[:16]}...")

(DRIVE_OUT / "assistant_v7_bhdr_report.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
if os.environ.get("COLAB_AUTO_UNASSIGN", "1") == "1":
    try:
        from google.colab import runtime
        print("[Colab] Auto-unassigning runtime cleanly...")
        runtime.unassign()
    except Exception as exc:
        print(f"[Note] Auto-unassign skipped: {exc}")